# 🌞 Solar Nowcast Loader
This notebook shows you how to load nowcasts and satellite observations with the validator.py:
1.  **Load** a time series of solar nowcasts.
2.  **Load** a time series of satellite observations.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path().resolve().parent))

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from validator import SatelliteNowcastLoader, SatelliteObservationLoader

NWC_DIR = Path("/dmidata/projects/energivejr/nowcasts")
OBS_DIR = Path("/dmidata/projects/energivejr/satellite_data")

nwc_start = "2026-02-26 11:00"

In [46]:
# --- Step 1: Load nowcasts ---
print(f"Loading nowcasts from {nwc_start}")
nwc_loader = SatelliteNowcastLoader(data_dir=NWC_DIR)
nowcast_ds = nwc_loader.load_data(nwc_start, nwc_start)
nowcast_ds

Loading nowcasts from 2026-02-26 11:00
  Found 1 nowcast files.


<xarray.Dataset> Size: 25MB
Dimensions:                  (initialization_time: 1, ensemble: 1,
                              lead_time: 24, lat: 368, lon: 713)
Coordinates:
  * initialization_time      (initialization_time) datetime64[us] 8B 2026-02-...
  * ensemble                 (ensemble) int64 8B 0
  * lead_time                (lead_time) timedelta64[ns] 192B 00:15:00 ... 06...
    valid_time               (initialization_time, lead_time) datetime64[ns] 192B ...
  * lat                      (lat) float64 3kB 63.48 63.43 63.39 ... 47.32 47.27
  * lon                      (lon) float64 6kB -10.73 -10.69 ... 19.94 19.98
Data variables:
    probabilistic_advection  (initialization_time, ensemble, lead_time, lat, lon) float32 25MB dask.array<chunksize=(1, 1, 24, 368, 713), meta=np.ndarray>
Attributes:
    description:    Simple Probabilistic Advection solar forecast using KNMI ...
    history:        Created 2026-02-26 11:23:04 UTC
    model_version:  0.3.0

In [47]:
# --- Step 2: Load observations ---
# Cover all valid_times: from first init_time to last init_time + max lead_time (6 h)
obs_start = pd.Timestamp(nwc_start)
obs_end   = pd.Timestamp(nwc_start) + pd.Timedelta(nowcast_ds.lead_time.max().values)

print(f"Loading observations from {obs_start} to {obs_end}...")
obs_loader = SatelliteObservationLoader(data_dir=OBS_DIR)
obs_ds = obs_loader.load_data(obs_start, obs_end)
obs_ds

Loading observations from 2026-02-26 11:00:00 to 2026-02-26 17:00:00...


<xarray.Dataset> Size: 52MB
Dimensions:  (time: 25, y: 368, x: 713)
Coordinates:
  * time     (time) datetime64[ns] 200B 2026-02-26T11:00:00 ... 2026-02-26T17...
  * y        (y) float64 3kB 63.48 63.43 63.39 63.35 ... 47.4 47.36 47.32 47.27
  * x        (x) float64 6kB -10.73 -10.69 -10.64 -10.6 ... 19.89 19.94 19.98
Data variables:
    sds      (time, y, x) float32 26MB dask.array<chunksize=(1, 368, 713), meta=np.ndarray>
    sds_cs   (time, y, x) float32 26MB dask.array<chunksize=(1, 368, 713), meta=np.ndarray>
Attributes: (12/25)
    Conventions:                                              CF-1.6
    title:                                                    Cloud Physical ...
    institution:                                              Royal Netherlan...
    source:                                                   CPP_6.1
    history:                                                  Created by ADAG...
    references:                                               http://msgcpp.k...
    ...                                                       ...
    source_surface_emissivity:                                CIMMS
    subsatellite_longitude_nominal:                           0.0
    subsatellite_longitude_actual:                            0.14881438
    subsatellite_latitude_actual:                             -4.344148
    adaguc_wcs_destgridspec:                                  width=713&heigh...
    software:                                                 ADAGUC WCS Serv...